In [3]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score


# --------------------------------------------------
# 1. Split the data and perform feature scaling
# --------------------------------------------------

def split_scalar(indep_X, dep_Y):

    X_train, X_test, y_train, y_test = train_test_split(
        indep_X,
        dep_Y,
        test_size=0.25,
        random_state=0
    )

    sc = StandardScaler()

    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    return X_train, X_test, y_train, y_test


# --------------------------------------------------
# 2. Calculate R2 score
# --------------------------------------------------

def r2_prediction(regressor, X_test, y_test):

    y_pred = regressor.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    return r2


# --------------------------------------------------
# 3. Linear Regression
# --------------------------------------------------

def Linear(X_train, y_train, X_test, y_test):

    regressor = LinearRegression()

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# --------------------------------------------------
# 4. Linear SVR
# --------------------------------------------------

def svm_linear(X_train, y_train, X_test, y_test):

    regressor = SVR(kernel='linear')

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# --------------------------------------------------
# 5. Non-linear SVR (RBF kernel)
# --------------------------------------------------

def svm_NL(X_train, y_train, X_test, y_test):

    regressor = SVR(kernel='rbf')

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# --------------------------------------------------
# 6. Decision Tree Regression
# --------------------------------------------------

def Decision(X_train, y_train, X_test, y_test):

    regressor = DecisionTreeRegressor(random_state=0)

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# --------------------------------------------------
# 7. Random Forest Regression
# --------------------------------------------------

def random(X_train, y_train, X_test, y_test):

    regressor = RandomForestRegressor(
        n_estimators=10,
        random_state=0
    )

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# --------------------------------------------------
# 8. Recursive Feature Elimination (RFE)
# --------------------------------------------------

def rfeFeature(indep_X, dep_Y, n):

    rfelist = []
    feature_names = []

    # Estimators supported by RFE
    rfemodellist = [
        LinearRegression(),
        SVR(kernel='linear'),
        DecisionTreeRegressor(random_state=0),
        RandomForestRegressor(
            n_estimators=10,
            random_state=0
        )
    ]

    for model in rfemodellist:

        print("\nRFE Model:", model)

        log_rfe = RFE(
            estimator=model,
            n_features_to_select=n
        )

        log_fit = log_rfe.fit(indep_X, dep_Y)

        # Get the selected feature names
        selected_features = indep_X.columns[
            log_fit.support_
        ].tolist()

        print("Selected Features:", selected_features)

        # Store the selected feature names
        feature_names.append(selected_features)

        # Store the feature-selected dataset
        log_rfe_feature = log_fit.transform(indep_X)

        rfelist.append(log_rfe_feature)

    return rfelist, feature_names


# --------------------------------------------------
# 9. Compare regression models
# --------------------------------------------------

def rfe_regression(
    acclog,
    accsvml,
    accsvmnl,
    accdes,
    accrf
):

    rfedataframe = pd.DataFrame(
        index=[
            'Linear Regression RFE',
            'Linear SVR RFE',
            'Decision Tree RFE',
            'Random Forest RFE'
        ],
        columns=[
            'Linear',
            'SVMl',
            'SVMnl',
            'Decision',
            'Random'
        ]
    )

    for number, index_name in enumerate(rfedataframe.index):

        rfedataframe.loc[index_name, 'Linear'] = acclog[number]

        rfedataframe.loc[index_name, 'SVMl'] = accsvml[number]

        rfedataframe.loc[index_name, 'SVMnl'] = accsvmnl[number]

        rfedataframe.loc[index_name, 'Decision'] = accdes[number]

        rfedataframe.loc[index_name, 'Random'] = accrf[number]

    return rfedataframe


# --------------------------------------------------
# 10. Load the dataset
# --------------------------------------------------

dataset1 = pd.read_csv("prep.csv", index_col=None)

df2 = dataset1.copy()

# Convert categorical columns into numeric dummy columns
df2 = pd.get_dummies(df2, drop_first=True)

# Separate input features and target
indep_X = df2.drop('classification_yes', axis=1)

dep_Y = df2['classification_yes']


# --------------------------------------------------
# 11. Check the data
# --------------------------------------------------

print("\nInput features:")
print(indep_X.columns.tolist())

print("\nTarget values:")
print(dep_Y.value_counts())

print("\nDataset shape:", df2.shape)


# --------------------------------------------------
# 12. Apply RFE to select three features
# --------------------------------------------------

rfelist, feature_names = rfeFeature(
    indep_X,
    dep_Y,
    3
)


# --------------------------------------------------
# 13. Create empty lists to store R2 scores
# --------------------------------------------------

acclin = []
accsvml = []
accsvmnl = []
accdes = []
accrf = []


# --------------------------------------------------
# 14. Evaluate all five regression models
# --------------------------------------------------

for i, selected_X in enumerate(rfelist):

    print("\n======================================")
    print("RFE Method:", feature_names[i])
    print("======================================")

    # Split and scale selected features
    X_train, X_test, y_train, y_test = split_scalar(
        selected_X,
        dep_Y
    )

    # Linear Regression
    r2_linear = Linear(
        X_train, y_train, X_test, y_test
    )
    accl_in_value = r2_linear


Input features:
['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal', 'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes', 'appet_yes', 'pe_yes', 'ane_yes']

Target values:
1    249
0    150
Name: classification_yes, dtype: int64

Dataset shape: (399, 28)

RFE Model: LinearRegression()
Selected Features: ['sg_c', 'sg_d', 'sg_e']

RFE Model: SVR(kernel='linear')
Selected Features: ['sg_c', 'sg_d', 'sg_e']

RFE Model: DecisionTreeRegressor(random_state=0)
Selected Features: ['hrmo', 'sg_c', 'sg_d']

RFE Model: RandomForestRegressor(n_estimators=10, random_state=0)
Selected Features: ['al', 'hrmo', 'sg_d']

RFE Method: ['sg_c', 'sg_d', 'sg_e']

RFE Method: ['sg_c', 'sg_d', 'sg_e']

RFE Method: ['hrmo', 'sg_c', 'sg_d']

RFE Method: ['al', 'hrmo', 'sg_d']
